# MARIDA Dataset Exploration

Quick visualization of one patch: RGB composite + spectral indices + label mask.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.dataset_loader import load_patch
from src.features.spectral_indices import compute_ndvi, compute_fdi, compute_ndwi

In [ ]:
# Pick any patch from data/raw/patches
PATCHES_ROOT = Path('../data/raw/patches')
patch_dirs = sorted([p for p in PATCHES_ROOT.iterdir() if p.is_dir()])
print(f'Found {len(patch_dirs)} patches')
print('First 5:', [p.name for p in patch_dirs[:5]])

In [ ]:
# Load one patch
bands, label = load_patch(str(patch_dirs[0]))
print('Bands shape:', bands.shape)   # (6, H, W)
print('Label shape:', label.shape)
print('Unique label values:', np.unique(label))

In [ ]:
B02, B03, B04, B08, B11, B12 = bands

# RGB composite (simple percentile stretch)
rgb = np.stack([B04, B03, B02], axis=-1)
p2, p98 = np.percentile(rgb, (2, 98))
rgb_stretched = np.clip((rgb - p2) / (p98 - p2 + 1e-9), 0, 1)

ndvi = compute_ndvi(B08, B04)
fdi  = compute_fdi(B08, B04, B11)
ndwi = compute_ndwi(B03, B08)

fig, ax = plt.subplots(2, 3, figsize=(14, 9))
ax[0, 0].imshow(rgb_stretched);          ax[0, 0].set_title('RGB composite')
ax[0, 1].imshow(ndvi, cmap='RdYlGn');    ax[0, 1].set_title('NDVI')
ax[0, 2].imshow(fdi,  cmap='magma');     ax[0, 2].set_title('FDI')
ax[1, 0].imshow(ndwi, cmap='Blues');     ax[1, 0].set_title('NDWI')
ax[1, 1].imshow(label, cmap='tab20');    ax[1, 1].set_title('Label mask')
ax[1, 2].imshow(label == 1, cmap='Reds'); ax[1, 2].set_title('Marine Debris (binary)')
for a in ax.ravel():
    a.axis('off')
plt.tight_layout()
plt.show()